# 03 — Exploratory Data Analysis
Distributions, correlations, outlier boxplots, age-group breakdowns.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')

Ready.


## 1. Load clean data

In [2]:
df = pd.read_csv('../../data/processed/bioage_final_clean.csv')
print(f'Clean data: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

Clean data: 19992 rows x 13 columns


,Age,LBXSAL,LBXSCR,LBXGLU,CRP,LBXLYPCT,LBXMCVSI,LBXRDW,LBXSAPSI,LBXWBCSI,LBXGH,LBDHDD,LBXTC
0,44.0,3.5,0.8,90.0,2.44,35.8,80.1,13.7,74.0,5.3,6.0,39.0,105.0
1,70.0,5.0,1.2,157.0,0.05,29.4,90.3,12.5,48.0,7.5,7.1,59.0,147.0
2,73.0,3.9,1.2,100.0,0.21,29.1,88.6,13.4,77.0,6.6,5.9,49.0,186.0
3,79.0,4.1,0.9,97.0,0.20,18.8,97.2,12.4,67.0,6.5,5.0,81.0,181.0
4,59.0,3.7,0.9,86.0,0.15,37.8,85.9,13.6,99.0,4.3,5.8,76.0,205.0


## 2. Feature distributions

In [3]:
feature_cols = [c for c in df.columns if c != 'Age']
n_features = len(feature_cols)
n_cols_plot = 3
n_rows_plot = (n_features + n_cols_plot - 1) // n_cols_plot

fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(14, n_rows_plot * 3.5))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=40, edgecolor='white', alpha=0.8, color='steelblue')
    ax.set_title(col, fontsize=11)
    skew = data.skew()
    ax.text(0.95, 0.95, f'skew={skew:.1f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions (Clean Data)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/06_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/06_distributions.png')

Saved: reports/figures/06_distributions.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_21168\505560087.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Correlation heatmap

In [4]:
corr = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation Matrix — All Features + Age')
plt.tight_layout()
plt.savefig('../../reports/figures/07_correlation_heatmap.png', dpi=150)
plt.show()
print('Saved: reports/figures/07_correlation_heatmap.png')

C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\matrix.py:260: FutureWarning: Format strings passed to MaskedConstant are ignored, but in future may error or produce different behavior
  annotation = ("{:" + self.fmt + "}").format(val)


Saved: reports/figures/07_correlation_heatmap.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_21168\1911911005.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Top correlations with Age
age_corr = corr['Age'].drop('Age').sort_values(key=abs, ascending=False)
print('Correlations with Age (sorted by |r|):')
print(age_corr.to_string())

Correlations with Age (sorted by |r|):
LBXGH       0.296605
LBXSCR      0.238509
LBXGLU      0.232609
LBXMCVSI    0.204473
LBXRDW      0.183826
LBXLYPCT   -0.174465
LBXSAL     -0.164590
LBXSAPSI    0.130186
LBXTC       0.095963
LBDHDD      0.073732
CRP         0.051221
LBXWBCSI   -0.043217


## 4. Boxplots per biomarker

In [6]:
fig, axes = plt.subplots(1, len(feature_cols), figsize=(4 * len(feature_cols), 5))
if len(feature_cols) == 1:
    axes = [axes]

for i, col in enumerate(feature_cols):
    ax = axes[i]
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red'))
    ax.set_title(col, fontsize=9, rotation=45)
    ax.set_xticklabels([])

fig.suptitle('Boxplots — Candidate Features', fontsize=14)
plt.tight_layout()
plt.savefig('../../reports/figures/08_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/08_boxplots.png')

Saved: reports/figures/08_boxplots.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_21168\1459248796.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Age-group breakdowns

In [7]:
bins = [0, 18, 30, 45, 60, 75, 100]
labels = ['0-17', '18-29', '30-44', '45-59', '60-74', '75+']
df['age_group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

# Top 4 features most correlated with Age
top4 = age_corr.head(4).index.tolist()
print(f'Top 4 Age-correlated features: {top4}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(top4):
    ax = axes[i // 2][i % 2]
    order = labels
    sns.boxplot(data=df, x='age_group', y=col, order=order, ax=ax,
                palette='viridis', fliersize=2)
    ax.set_title(f'{col} by Age Group')
    ax.set_xlabel('Age Group')

fig.suptitle('Top Biomarkers Stratified by Age Group', fontsize=14)
plt.tight_layout()
plt.savefig('../../reports/figures/09_age_group_breakdowns.png', dpi=150)
plt.show()
print('Saved: reports/figures/09_age_group_breakdowns.png')

C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)


Top 4 Age-correlated features: ['LBXGH', 'LBXSCR', 'LBXGLU', 'LBXMCVSI']


C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain

Saved: reports/figures/09_age_group_breakdowns.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_21168\613287069.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Feature vs Age scatter plots

In [8]:
n_feat = len(feature_cols)
n_cols = 3
n_rows = (n_feat + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.array(axes).flatten() if n_feat > 1 else [axes]

for i, col in enumerate(feature_cols):
    ax = axes[i]
    ax.scatter(df['Age'], df[col], alpha=0.15, s=5, color='steelblue')
    # Add regression line
    mask = df[col].notna() & df['Age'].notna()
    if mask.sum() > 10:
        slope, intercept, r, p, se = stats.linregress(df.loc[mask, 'Age'], df.loc[mask, col])
        x_line = np.linspace(df['Age'].min(), df['Age'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='red', linewidth=2,
                label=f'r={r:.3f}')
        ax.legend(fontsize=9)
    ax.set_xlabel('Age')
    ax.set_ylabel(col)
    ax.set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Features vs Age with Linear Fit', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/10_feature_vs_age.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/10_feature_vs_age.png')

Saved: reports/figures/10_feature_vs_age.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_21168\3463161576.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Summary statistics table

In [9]:
summary = df.describe().round(3).T
summary['skew'] = df[feature_cols + ['Age']].skew().round(2)
summary['kurtosis'] = df[feature_cols + ['Age']].kurtosis().round(2)
summary

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
Age,19992.0,49.092,18.515,18.00,33.00,50.00,64.00,80.0,-0.02,-1.16
LBXSAL,19992.0,4.122,0.367,1.50,3.90,4.10,4.40,5.5,-0.47,1.21
LBXSCR,19992.0,0.880,0.326,0.30,0.70,0.83,1.00,5.0,5.63,59.81
LBXGLU,19992.0,110.236,36.293,40.00,94.00,101.00,112.00,500.0,4.20,23.43
CRP,19992.0,2.673,6.570,0.01,0.25,0.85,2.65,188.5,9.71,150.80
LBXLYPCT,19992.0,30.756,8.752,2.90,24.60,30.30,36.30,80.0,0.40,0.59
LBXMCVSI,19992.0,88.838,5.922,50.80,85.90,89.30,92.40,120.0,-0.90,3.27
LBXRDW,19992.0,13.483,1.366,9.70,12.60,13.20,14.00,25.0,2.33,10.37
LBXSAPSI,19992.0,75.488,26.710,16.00,58.00,71.00,87.00,500.0,2.63,20.47
LBXWBCSI,19992.0,6.804,2.134,1.60,5.40,6.50,7.90,50.0,2.73,33.66


## EDA Summary
See saved charts in `reports/figures/` for presentation. Key findings:
1. **Age is top-coded at 80** — spike removed during cleaning.
2. **Right-skewed features**: CRP, LBXSAPSI, LBXWBCSI — log transforms may help.
3. **Strongest Age correlations**: [to be filled after running].
4. **Multicollinearity**: some CBC-derived features are highly correlated (r > 0.8).
5. **Age-group patterns**: biomarker distributions shift with age, some non-linearly.
